In [99]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import resnet18
import timm
import numpy as np
import os
from PIL import Image
from collections import Counter
from scipy.io import loadmat

In [100]:
import os
from glob import glob

root_dir = r"L:\常惠林\萎凋\自然萎凋\原始"
image_count = sum(len(files) for _, _, files in os.walk(root_dir))
print("图像数量:", image_count)
print("NIR 数据 shape:", nir_data.shape)


图像数量: 120
NIR 数据 shape: torch.Size([120, 128])


In [ ]:
# ========== 数据加载与增强 ==========
def load_nir_from_mat(mat_path):
    data = loadmat(mat_path)
    for key in ['nir', 'NIR', 'data', 'spectra', 'X']:
        if key in data:
            return torch.tensor(data[key], dtype=torch.float32)
    for key, value in data.items():
        if isinstance(value, np.ndarray) and value.ndim == 2:
            return torch.tensor(value, dtype=torch.float32)
    raise ValueError("No valid NIR data found in .mat file")

class SimCLRDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img_path = self.base_dataset.image_paths[idx]
        nir = self.base_dataset.nir_data[idx]
        label = self.base_dataset.labels[idx]

        img = Image.open(img_path).convert("RGB")
        x1 = self.transform(img)
        x2 = self.transform(img)

        # 返回两个 view + nir + label
        return x1, x2, torch.tensor(nir, dtype=torch.float32), torch.tensor(label)

      


In [102]:
# ========== SimCLR 模型与损失 ==========
class SimCLRStudent(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        base = resnet18(pretrained=False)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.projection = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, out_dim))
    def forward(self, x):
        h = self.encoder(x).squeeze()
        return self.projection(h)

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    def forward(self, z1, z2):
        z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
        reps = torch.cat([z1, z2], dim=0)
        sim = F.cosine_similarity(reps.unsqueeze(1), reps.unsqueeze(0), dim=2)
        batch_size = z1.size(0)
        mask = torch.eye(2*batch_size, dtype=torch.bool).to(z1.device)
        labels = torch.cat([torch.arange(batch_size) for _ in range(2)], dim=0)
        labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        labels = labels[~mask].view(2*batch_size, -1)
        sim = sim[~mask].view(2*batch_size, -1)
        positives = sim[labels.bool()].view(labels.shape[0], -1)
        negatives = sim[~labels.bool()].view(sim.shape[0], -1)
        logits = torch.cat([positives, negatives], dim=1) / self.temperature
        return F.cross_entropy(logits, torch.zeros(logits.size(0), dtype=torch.long).to(z1.device))

In [103]:
# ========== 下游蒸馏模型 ==========
class NIR_Encoder(nn.Module):
    def __init__(self, input_dim=1, output_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, output_dim))
    def forward(self, x):
        return self.encoder(x)

class ContrastiveFusion(nn.Module):
    def __init__(self, img_dim=512, nir_dim=128, out_dim=256):
        super().__init__()
        self.img_proj = nn.Linear(img_dim, out_dim)
        self.nir_proj = nn.Linear(nir_dim, out_dim)
    def forward(self, img_feat, nir_feat):
        return self.img_proj(img_feat) + self.nir_proj(nir_feat)

class TeacherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('resnet50', pretrained=True, num_classes=0)
    def forward(self, x):
        return self.encoder(x)

class MultiModalDistillationModel(nn.Module):
    def __init__(self, student_encoder, teacher_encoder, nir_encoder, fusion, num_classes):
        super().__init__()
        self.student_encoder = student_encoder
        self.teacher_encoder = teacher_encoder
        self.nir_encoder = nir_encoder
        self.fusion = fusion
        self.classifier = nn.Sequential(nn.LayerNorm(256), nn.ReLU(), nn.Linear(256, num_classes))
    def forward(self, img, nir):
        student_feat = self.student_encoder(img)
        with torch.no_grad():
            teacher_feat = self.teacher_encoder(img)
        nir_feat = self.nir_encoder(nir)
        fused_feat = self.fusion(student_feat, nir_feat)
        return self.classifier(fused_feat), student_feat, teacher_feat

class PureDistillationLoss(nn.Module):
    def __init__(self, temperature=3.0):
        super().__init__()
        self.temperature = temperature
    def forward(self, student_logits, teacher_logits):
        return F.kl_div(F.log_softmax(student_logits/self.temperature, dim=1),
                        F.softmax(teacher_logits/self.temperature, dim=1), reduction='batchmean')

In [ ]:
# ========== 训练流程 ==========
def train_simclr(student, loader, epochs=10):
    criterion = nn.CosineEmbeddingLoss()
    optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

    for epoch in range(epochs):
        total_loss = 0
        for x1, x2, nir, label in loader:
            x1 = x1.cuda()
            x2 = x2.cuda()
            nir = nir.cuda()
            label = label.cuda()

            z1 = student(x1)
            z2 = student(x2)

            # MoCo-style: 可加入 NIR 特征（如拼接或投影后融合）
            # 示例：简单拼接后再计算相似度
            combined1 = torch.cat([z1, nir], dim=1)
            combined2 = torch.cat([z2, nir], dim=1)

            # cosine loss
            y = torch.ones(x1.size(0)).to(x1.device)
            loss = criterion(combined1, combined2, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] Loss: {total_loss/len(loader):.4f}")


def train_pure_distill(model, teacher, loader, epochs, optimizer, loss_fn):
    model.train()
    teacher.eval()
    for epoch in range(epochs):
        total_loss = 0
        for imgs, nirs, _ in simclr_loader:
            imgs, nirs = imgs.cuda(), nirs.cuda()
            optimizer.zero_grad()
            logits, _, _ = model(imgs, nirs)
            with torch.no_grad():
                teacher_logits = teacher(imgs)
            loss = loss_fn(logits, teacher_logits)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Distillation Epoch {epoch+1}: Loss = {total_loss/len(loader):.4f}")

In [105]:
# ========== 主程序 ==========
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    nir_data = load_nir_from_mat(r"L:\\常惠林\\萎凋\\NIR.mat")
    base_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    simclr_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    dataset = EnhancedDataset(r"L:\\常惠林\\萎凋\\自然萎凋\\原始", nir_data, base_transform)
    simclr_loader = DataLoader(SimCLRDataset(dataset, simclr_transform), batch_size=64, shuffle=True)
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

    simclr_student = SimCLRStudent().to(device)
    print("预训练 student (SimCLR)...")
    train_simclr(simclr_student, simclr_loader, epochs=10)

    student_encoder = simclr_student.encoder
    teacher_encoder = TeacherModel().to(device)
    nir_encoder = NIR_Encoder(input_dim=nir_data.shape[1]).to(device)
    fusion = ContrastiveFusion().to(device)
    model = MultiModalDistillationModel(student_encoder, teacher_encoder, nir_encoder, fusion, num_classes=3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = PureDistillationLoss()

    print("开始下游蒸馏训练...")
    train_pure_distill(model, teacher_encoder, train_loader, epochs=10, optimizer=optimizer, loss_fn=loss_fn)

预训练 student (SimCLR)...


c:\Users\enine\anaconda3\envs\changhl\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\enine\anaconda3\envs\changhl\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)